# Day 4: scientific-audit functions

Completed workflow generated from `course_code/`; edit the canonical Python modules and rebuild with `python tools/build_course.py`.

Run setup, definition cells, then the workflow cell. Each notebook contains its Python dependencies; opening it does not clone the repository. This is controlled simulated DNA classification, not real promoter discovery. Student follow-along is optional pending the instructor's delivery choice.

See [execution and access instructions](https://github.com/palexiou/ML_Course_2026/blob/main/notebooks/README.md) and the separate scaffolded practicals and instructor answers. Save a Drive copy to retain edits.


In [ ]:
# Install missing direct dependencies only; existing packages are not silently replaced.
# Exact locally rehearsed versions below are a baseline, not verified Colab compatibility.
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys
requirements = ['numpy==2.3.2', 'pandas==2.3.1', 'scikit-learn==1.7.2', 'matplotlib==3.10.5']
missing = []
for requirement in requirements:
    package, expected = requirement.split('==')
    try:
        installed = version(package)
        print(f'{package}: {installed} (local baseline {expected})')
    except PackageNotFoundError:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    print('If prompted, restart the runtime, then rerun this notebook from the top.')


In [ ]:
"""Shared simulation, evaluation, and artifact contracts for the course."""

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

BASE_ORDER = "ACGT"
PRIMARY_GENERATOR_VERSION = "independent-ppm15-forward-v2"
DATASET_SCHEMA_VERSION = 2
# Rows are positions, columns A/C/G/T. Fixed independently of the dataset seed.
DEFAULT_MOTIF_PPM = (
    (0.05, 0.05, 0.10, 0.80),
    (0.70, 0.10, 0.10, 0.10),
    (0.05, 0.05, 0.15, 0.75),
    (0.65, 0.15, 0.10, 0.10),
    (0.50, 0.40, 0.05, 0.05),
    (0.60, 0.10, 0.20, 0.10),
    (0.10, 0.10, 0.10, 0.70),
    (0.10, 0.10, 0.70, 0.10),
    (0.10, 0.65, 0.15, 0.10),
    (0.15, 0.10, 0.65, 0.10),
    (0.55, 0.15, 0.20, 0.10),
    (0.10, 0.60, 0.10, 0.20),
    (0.10, 0.10, 0.75, 0.05),
    (0.10, 0.10, 0.15, 0.65),
    (0.15, 0.55, 0.20, 0.10),
)

@dataclass(frozen=True)
class SimulationConfig:
    seed: int = 2026
    sample_count: int = 1200
    sequence_length: int = 160
    # Array inputs are copied into immutable tuples during validation.
    motif_ppm: tuple[tuple[float, ...], ...] | np.ndarray = DEFAULT_MOTIF_PPM
    positive_plant_probability: float = 0.95
    negative_plant_probability: float = 0.05
    kmer_length: int = 4

    def __post_init__(self) -> None:
        if not 0 <= self.seed < 2**32:
            raise ValueError("seed must be between 0 and 2**32 - 1")
        if self.sample_count < 20 or self.sample_count % 2:
            raise ValueError("sample_count must be an even integer of at least 20")
        matrix = np.array(self.motif_ppm, dtype=float, copy=True)
        if matrix.shape != (15, 4):
            raise ValueError("motif_ppm must have shape (15 positions, 4 ACGT bases)")
        if not np.isfinite(matrix).all() or np.any(matrix <= 0) or np.any(matrix >= 1):
            raise ValueError("motif_ppm entries must be finite, strictly positive and less than one")
        if not np.allclose(matrix.sum(axis=1), 1.0, rtol=0, atol=1e-12):
            raise ValueError("motif_ppm rows must sum to one")
        if np.any(matrix.max(axis=1) < 0.5):
            raise ValueError("Each motif_ppm position needs a leading probability of at least 0.5")
        object.__setattr__(self, "motif_ppm", tuple(tuple(float(p) for p in row) for row in matrix))
        if self.sequence_length < len(self.motif_ppm):
            raise ValueError("sequence_length must be at least the motif length")
        if not 1 <= self.kmer_length <= self.sequence_length:
            raise ValueError("kmer_length must be between 1 and sequence_length")
        for probability in (
            self.positive_plant_probability,
            self.negative_plant_probability,
        ):
            if not 0 <= probability <= 1:
                raise ValueError("motif planting probabilities must be between 0 and 1")


def simulation_settings(config: SimulationConfig) -> dict[str, Any]:
    return {"dataset_variant": "simple_signal", "algorithm_version": PRIMARY_GENERATOR_VERSION,
            "base_order": BASE_ORDER, "background_probabilities": [0.25] * 4,
            "orientation": "forward_only", **asdict(config)}


def settings_fingerprint(settings: dict[str, Any]) -> str:
    return hashlib.sha256(json.dumps(settings, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def generation_contract() -> dict[str, Any]:
    return {"schema_version": DATASET_SCHEMA_VERSION,
            "primary_defaults": simulation_settings(SimulationConfig())}


def expected_motif_kmers(config: SimulationConfig, words: Any) -> pd.DataFrame:
    """Average internal-window probability; enrichment denominator is uniform 4^-k.

    Excludes windows crossing site boundaries, label mixtures and full-sequence counts.
    For k > 15 there are no wholly internal windows, so the annotation is undefined.
    """
    k = config.kmer_length
    if k > 15:
        return pd.DataFrame(columns=["kmer", "motif_window_probability", "motif_vs_background_enrichment"])
    matrix = np.asarray(config.motif_ppm)
    records = []
    for word in words:
        if len(word) != k or set(word) - set(BASE_ORDER):
            raise ValueError("Expected ACGT words of the configured k-mer length")
        bases = [BASE_ORDER.index(base) for base in word]
        probability = sum(float(np.prod(matrix[start + np.arange(k), bases]))
                          for start in range(16 - k)) / (16 - k)
        records.append({"kmer": word,
                        "motif_window_probability": probability,
                        "motif_vs_background_enrichment": probability / (0.25 ** k)})
    return pd.DataFrame(records)


def sample_motif_site(random: np.random.Generator, config: SimulationConfig) -> str:
    # Independent draws for every position of every inserted site.
    return "".join(random.choice(list(BASE_ORDER), p=row) for row in config.motif_ppm)


def generate_course_dataset(
    config: SimulationConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    random = np.random.default_rng(config.seed)
    labels = np.repeat([0, 1], config.sample_count // 2)
    random.shuffle(labels)
    sequences = random.choice(list("ACGT"), size=(config.sample_count, config.sequence_length))
    planting_probabilities = np.where(
        labels == 1,
        config.positive_plant_probability,
        config.negative_plant_probability,
    )
    planted = random.random(config.sample_count) < planting_probabilities
    starts = np.full(config.sample_count, -1, dtype=int)
    sites: list[str | None] = [None] * config.sample_count
    for index in np.flatnonzero(planted):
        start = int(random.integers(0, config.sequence_length - len(config.motif_ppm) + 1))
        site = sample_motif_site(random, config)
        sequences[index, start : start + len(config.motif_ppm)] = list(site)
        starts[index] = start
        sites[index] = site
    sample_ids = [f"simple-{config.seed}-{index:05d}" for index in range(config.sample_count)]
    samples = pd.DataFrame(
        {
            "sample_id": sample_ids,
            "group_id": sample_ids,
            "sequence": ["".join(sequence) for sequence in sequences],
            "label": labels,
        }
    )
    if samples["sequence"].duplicated().any():
        raise ValueError("The independent-sample variant contains duplicate sequences")
    truth = pd.DataFrame(
        {"sample_id": sample_ids, "motif_planted": planted,
         "motif_start": pd.array([int(start) if flag else None for start, flag in zip(starts, planted)], dtype="Int64"),
         "sampled_site": sites}
    )
    return samples, truth


def assign_splits(samples: pd.DataFrame, seed: int) -> pd.DataFrame:
    train_indices, heldout_indices = train_test_split(
        samples.index,
        test_size=0.4,
        stratify=samples["label"],
        random_state=seed,
    )
    validation_indices, test_indices = train_test_split(
        heldout_indices,
        test_size=0.5,
        stratify=samples.loc[heldout_indices, "label"],
        random_state=seed,
    )
    result = samples.copy()
    result["split"] = "train"
    result.loc[validation_indices, "split"] = "validation"
    result.loc[test_indices, "split"] = "test"
    if result.groupby("group_id")["split"].nunique().max() != 1:
        raise ValueError("Related sequences cross split boundaries")
    if len(train_indices) + len(validation_indices) + len(test_indices) != len(result):
        raise ValueError("Splits do not cover the course dataset")
    return result



def classification_metrics(labels: Any, probabilities: Any, evaluation_split: str) -> dict[str, Any]:
    if evaluation_split not in {"validation", "test"}:
        raise ValueError("Evaluation split must be validation or test")
    if not np.isfinite(probabilities).all() or np.any(np.asarray(probabilities) < 0) or np.any(np.asarray(probabilities) > 1):
        raise ValueError("Probabilities must be finite and between zero and one")
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    return {
        "evaluation_split": evaluation_split, "primary_metric": "roc_auc", "decision_threshold": 0.5,
        "sample_count": len(labels), "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }


def validation_metrics(labels: Any, probabilities: Any) -> dict[str, Any]:
    return classification_metrics(labels, probabilities, "validation")


def write_json(path: str | Path, value: Any) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def file_hash(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def sample_hash(samples: pd.DataFrame) -> str:
    return hashlib.sha256(samples.to_csv(index=False, lineterminator="\n").encode()).hexdigest()


def save_dataset(root: str | Path, samples: pd.DataFrame, truth: pd.DataFrame, settings: dict[str, Any]) -> None:
    if "motif" in settings or ("motif_ppm" in settings and settings.get("algorithm_version") != PRIMARY_GENERATOR_VERSION):
        raise ValueError("Legacy primary settings: regenerate using simulation_settings(config)")
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    for split, filename in (("train", "train.csv"), ("validation", "validation.csv"), ("test", "protected_test.csv")):
        samples.loc[samples["split"] == split].to_csv(root / filename, index=False)
    samples[["sample_id", "group_id", "split"]].to_csv(root / "split_manifest.csv", index=False)
    truth.to_csv(root / "simulation_truth.csv", index=False)
    write_json(root / "generation.json", {
        "schema_version": DATASET_SCHEMA_VERSION, "settings": settings,
        "settings_sha256": settings_fingerprint(settings), "dataset_sha256": sample_hash(samples),
        "files": {name: file_hash(root / name) for name in
                  ("train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv")},
    })


def validate_generation(root: str | Path) -> dict[str, Any]:
    """Reject legacy generation and tampered data before development/freeze reads."""
    root = Path(root)
    metadata = json.loads((root / "generation.json").read_text(encoding="utf-8"))
    if metadata.get("schema_version") != DATASET_SCHEMA_VERSION:
        raise ValueError("Incompatible dataset metadata; regenerate development artifacts in a new directory")
    settings = metadata["settings"]
    if "motif" in settings:
        raise ValueError("Legacy fixed-motif dataset; regenerate with the PPM generator")
    if settings.get("dataset_variant") == "simple_signal" or "motif_ppm" in settings:
        fields = SimulationConfig.__dataclass_fields__
        config = SimulationConfig(**{key: settings[key] for key in fields})
        if settings_fingerprint(settings) != settings_fingerprint(simulation_settings(config)):
            raise ValueError("Incompatible primary generator algorithm or settings")
    if metadata.get("settings_sha256") != settings_fingerprint(settings):
        raise ValueError("Generation settings fingerprint mismatch")
    names = {"train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv"}
    if set(metadata.get("files", {})) != names:
        raise ValueError("Incomplete dataset file identity metadata")
    for name in names:
        if file_hash(root / name) != metadata["files"][name]:
            raise ValueError(f"Dataset file identity mismatch: {name}")
    return metadata


def read_dataset(root: str | Path, include_test_labels: bool = False) -> pd.DataFrame:
    root = Path(root)
    validate_generation(root)
    partitions = []
    for split, filename in (("train", "train.csv"), ("validation", "validation.csv"), ("test", "protected_test.csv")):
        if split == "test" and not include_test_labels:
            partition = pd.read_csv(root / filename, usecols=lambda column: column != "label")
            partition["label"] = np.nan
        else:
            partition = pd.read_csv(root / filename)
        required = {"sample_id", "group_id", "sequence", "label", "split"}
        non_null = required - ({"label"} if split == "test" and not include_test_labels else set())
        if not required <= set(partition) or partition[list(non_null)].isna().any().any():
            raise ValueError("Dataset columns are missing or contain null values")
        labels_expected = split != "test" or include_test_labels
        if partition.empty or not partition["split"].eq(split).all() or (labels_expected and set(partition["label"]) != {0, 1}):
            raise ValueError("A partition has an incorrect split or lacks both binary classes")
        partitions.append(partition)
    samples = pd.concat(partitions, ignore_index=True)
    if not samples["sample_id"].is_unique or not samples["sequence"].map(lambda sequence: bool(sequence) and set(sequence) <= set("ACGT")).all():
        raise ValueError("Dataset IDs or DNA sequences are invalid")
    if samples.groupby("group_id")["split"].nunique().max() != 1:
        raise ValueError("Related sequences cross split boundaries")
    if samples.groupby("sequence")["split"].nunique().max() != 1:
        raise ValueError("Duplicate DNA sequences cross split boundaries")
    manifest = pd.read_csv(root / "split_manifest.csv")
    expected = samples[["sample_id", "group_id", "split"]].sort_values("sample_id").reset_index(drop=True)
    actual = manifest.sort_values("sample_id").reset_index(drop=True)
    if not expected.equals(actual):
        raise ValueError("Split manifest does not match sample identities")
    samples.attrs["dataset_generation_sha256"] = file_hash(root / "generation.json")
    return samples


def build_interaction_dataset(seed: int = 2026, pair_count: int = 240) -> tuple[pd.DataFrame, pd.DataFrame]:
    if pair_count < 20:
        raise ValueError("At least 20 independent pairs are required")
    random = np.random.default_rng(seed)
    motif_first, motif_second = "TATAAATG", "CGGACGTC"
    flank_left, flank_right = "GGGGGG", "CCCCCC"
    positions = (32, 80)
    records, truth = [], []
    for pair_index in range(pair_count):
        background = random.choice(list("ACGT"), size=160)
        group = f"order-{seed}-{pair_index:05d}"
        for label, motifs in ((1, (motif_first, motif_second)), (0, (motif_second, motif_first))):
            sequence = background.copy()
            for position, motif in zip(positions, motifs):
                block = flank_left + motif + flank_right
                sequence[position:position + len(block)] = list(block)
            sample_id = f"{group}-{label}"
            records.append({"sample_id": sample_id, "group_id": group, "sequence": "".join(sequence), "label": label})
            truth.append({"sample_id": sample_id, "first_block_start": positions[0], "second_block_start": positions[1], "first_motif": motifs[0], "second_motif": motifs[1]})
    samples = pd.DataFrame(records)
    groups = samples["group_id"].unique()
    train_groups, heldout_groups = train_test_split(groups, test_size=0.4, random_state=seed)
    validation_groups, test_groups = train_test_split(heldout_groups, test_size=0.5, random_state=seed)
    samples["split"] = "train"
    samples.loc[samples["group_id"].isin(validation_groups), "split"] = "validation"
    samples.loc[samples["group_id"].isin(test_groups), "split"] = "test"
    return samples, pd.DataFrame(truth)


## Stage 06: Day 4: scientific-audit functions

Run the definition cell, then the workflow cell when the instructor reaches this stage. Definitions do not train models. Generated outputs are temporary runtime files.


In [ ]:
"""Recompute prediction evidence without mistaking it for a complete scientific audit."""

from pathlib import Path
from typing import Any
import argparse
import json

import numpy as np
import pandas as pd



def audit_predictions(samples: pd.DataFrame, predictions: pd.DataFrame, evaluation_split: str = "validation", claimed_metrics: dict[str, Any] | None = None) -> dict[str, Any]:
    required = {"sample_id", "group_id", "sequence", "label", "split"}
    if not required <= set(samples) or not {"sample_id", "probability"} <= set(predictions):
        raise ValueError("Missing dataset or prediction columns")
    if samples[list(required - {"label"})].isna().any().any() or not samples["sample_id"].is_unique:
        raise ValueError("Dataset identities must be unique and non-null")
    if not set(samples["split"]) <= {"train", "validation", "test"} or not set(samples["label"].dropna()) <= {0, 1}:
        raise ValueError("Unexpected split names or binary labels")
    if evaluation_split not in {"validation", "test"}:
        raise ValueError("Audit only validation or explicitly supplied test predictions")
    selected = samples.loc[samples["split"] == evaluation_split]
    if selected["label"].isna().any() or set(selected["label"]) != {0, 1}:
        raise ValueError("The requested evaluation split must contain both binary labels")
    if selected.empty or not predictions["sample_id"].is_unique or set(predictions["sample_id"]) != set(selected["sample_id"]):
        raise ValueError("Prediction IDs must cover the requested evaluation split exactly once")
    joined = selected.merge(predictions[["sample_id", "probability"]], on="sample_id", validate="one_to_one")
    probabilities = joined["probability"].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("Predictions must be finite probabilities between zero and one")
    metrics = classification_metrics(joined["label"], probabilities, evaluation_split)
    overlapping_groups = samples.groupby("group_id")["split"].nunique()
    duplicate_sequences = samples.groupby("sequence")["split"].nunique()
    disagreements = []
    for metric, claimed in (claimed_metrics or {}).items():
        if metric not in metrics:
            disagreements.append(f"Unsupported metric claim: {metric}")
        elif isinstance(metrics[metric], (int, float)) and isinstance(claimed, (int, float)):
            if not np.isclose(metrics[metric], claimed, atol=1e-6, rtol=1e-6):
                disagreements.append(f"Claimed {metric}={claimed}; recomputed {metrics[metric]}")
        elif metrics[metric] != claimed:
            disagreements.append(f"Claimed {metric} differs from recomputed evidence")
    return {"metrics": metrics, "cross_split_group_count": int((overlapping_groups > 1).sum()), "cross_split_duplicate_sequence_count": int((duplicate_sequences > 1).sum()), "claim_disagreements": disagreements, "requires_manual_review": ["Is the independent group definition scientifically justified?", "Were all fitted preprocessing and model choices restricted to training and validation?", "Was the test workflow frozen before the planned reveal?", "Does model and embedding provenance match the claimed inputs?", "Do biological conclusions exceed the simulated task?"], "scope": "Recomputed evidence and identity checks only; not proof of correct training or biological validity"}


In [ ]:
print("Audit functions loaded. Call audit_predictions(samples, predictions).")


## Day 4 · Scientific audit, with or without an agent account

**GitHub → Open in Colab → setup → shared definitions → stage 06 definitions
→ these checkpoints.** Instructor demonstration or optional follow-along;
no account with an agent service, new model training, submission or homework.
Pause at each STOP. The notebook is self-contained: opening GitHub does not clone files.

First inspect an **authored flawed fixture**. Its probabilities are invented,
not a recorded model run. Later, recompute saved predictions from the genuine
**October 26 historical fixed-motif agent runs**. Those data differ from today's
15-position PPM task; scores are not a comparison with the current Days 1–3 models.
Protected course test files/results are never included or opened.


In [ ]:
"""DELIBERATELY FLAWED, MANUALLY AUTHORED FIXTURE; not an agent run."""

from io import StringIO

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


samples = pd.read_csv(StringIO("""sample_id,group_id,sequence,label,split
train-01,group-a,ACGTACGT,0,train
train-02,group-b,TATAAATG,1,train
validation-01,group-a,ACGTACGA,0,validation
validation-02,group-c,TATAAATA,1,validation
"""))
predictions = pd.read_csv(StringIO("""sample_id,probability
validation-02,0.1
validation-01,0.9
"""))
training = samples.loc[samples["split"] == "train"]
validation = samples.loc[samples["split"] == "validation"]
vectorizer = CountVectorizer(analyzer="char", ngram_range=(4, 4), lowercase=False)
vectorizer.fit(samples["sequence"])
assert set(training["sample_id"]).isdisjoint(validation["sample_id"])
reported_accuracy = (validation["label"].to_numpy() == (predictions["probability"].to_numpy() >= 0.5)).mean()
print("MANUALLY AUTHORED, DELIBERATELY FLAWED FIXTURE; probabilities are invented.")
print("Misleading positional accuracy:", reported_accuracy)
print("Authored unsupported claim: unique sample IDs prove no leakage and this predicts real promoters.")


**STOP · Section 2:** inspect `samples` and `predictions` above before auditing.
Which group crosses the split? Where was the vocabulary fitted? Which sample
owns each probability? The fixture has no fitted classifier behind these numbers.
Its claim of 100% positional accuracy is deliberately misleading.

At **Section 5**, continue below. If watching the demonstration, discuss the
tables with a partner. There is no scored quiz.


In [ ]:
d4_training = samples.loc[samples.split.eq("train")]
d4_validation = samples.loc[samples.split.eq("validation")]
d4_joined = d4_validation[["sample_id", "label"]].merge(predictions, on="sample_id", validate="one_to_one")
d4_positional = float((d4_validation.label.to_numpy() == (predictions.probability.to_numpy() >= 0.5)).mean())
d4_aligned = float((d4_joined.label == (d4_joined.probability >= 0.5)).mean())
d4_fixture_report = audit_predictions(samples, predictions, claimed_metrics={"accuracy": 1.0})
assert d4_positional == 1.0 and d4_aligned == 0.0
assert d4_fixture_report["cross_split_group_count"] == 1
assert d4_fixture_report["claim_disagreements"]
print("AUTHORED FIXTURE, not an observed agent result")
print("Cross-split groups:", sorted(set(d4_training.group_id) & set(d4_validation.group_id)))
print("Misleading positional accuracy:", d4_positional, "ID-aligned accuracy:", d4_aligned)
print(d4_joined.to_string(index=False))
print("Metric disagreement:", d4_fixture_report["claim_disagreements"])
print("MANUAL code inspection: vectorizer.fit(samples['sequence']) includes validation rows.")
print("The prediction-table audit cannot infer preprocessing scope or prove whether a test was accessed.")
print("Repairing the join does not repair group leakage or create a credible model comparison.")


### Reject invalid prediction tables

Reversing row order should leave ID-aligned metrics unchanged. Missing, duplicate,
unexpected IDs and nonfinite/out-of-range probabilities must be rejected.
These are in-memory corruptions of the small fixture, never protected data.


In [ ]:
d4_reordered = audit_predictions(samples, predictions.iloc[::-1])
assert d4_reordered["metrics"] == d4_fixture_report["metrics"]
d4_invalid = {
    "duplicate ID": pd.concat([predictions, predictions.iloc[:1]], ignore_index=True),
    "missing ID": predictions.iloc[:1].copy(),
    "unexpected ID": predictions.assign(sample_id=["unexpected", "validation-01"]),
    "nonfinite probability": predictions.assign(probability=[np.nan, 0.9]),
    "out-of-range probability": predictions.assign(probability=[1.2, 0.9]),
}
for d4_name, d4_bad in d4_invalid.items():
    try:
        audit_predictions(samples, d4_bad)
    except ValueError as d4_error:
        print("Rejected", d4_name, "→", d4_error)
    else:
        raise AssertionError("Invalid predictions accepted: " + d4_name)
print("Shuffling valid rows preserves the ID-aligned result.")


### Genuine previous runs · October 26, 2026

The following compressed payload contains the exact archived **training and
validation CSVs, saved predictions and parent audit**. It contains no test file,
model weights or executable remote code. Hashes identify the supplied bytes;
they do not establish correct training. Git has stored LF line endings; the original Windows input hashes are matched separately by reconstructing CRLF. Both agents received identical development
files; the two model choices and search spaces differ.

Raw source, full prompts, reports and intervention notes remain in
[the evidence packet](https://github.com/palexiou/ML_Course_2026/tree/main/agent_workshop/actual_runs/2026-10-26).
The vague run used a **disclosed motif prior**, not biological discovery.
Recomputing historical saved predictions does not rerun an agent or fit a model.


In [ ]:
import base64, hashlib, zlib
from io import StringIO
# Exact development-only archive bytes; decoded data is never executed.
d4_payload_b64 = '...'
d4_archive = json.loads(zlib.decompress(base64.b64decode(d4_payload_b64)))
for d4_name, d4_text in d4_archive["files"].items():
    assert hashlib.sha256(d4_text.encode()).hexdigest() == d4_archive["sha256"][d4_name]
d4_historical_train = pd.read_csv(StringIO(d4_archive["files"]["vague/train.csv"]))
d4_historical_validation = pd.read_csv(StringIO(d4_archive["files"]["vague/validation.csv"]))
d4_historical = pd.concat([d4_historical_train, d4_historical_validation], ignore_index=True)
assert d4_historical_train.split.eq("train").all() and len(d4_historical_train) == 720
assert d4_historical_validation.split.eq("validation").all() and len(d4_historical_validation) == 240
assert set(d4_historical.split) == {"train", "validation"}
d4_parent = json.loads(d4_archive["files"]["parent_audit.json"])
for d4_name in ["train.csv", "validation.csv"]:
    assert hashlib.sha256(d4_archive["files"]["vague/" + d4_name].replace("\r\n", "\n").replace("\n", "\r\n").encode()).hexdigest() == d4_parent["input_hashes"][d4_name]
print("Historical fixed-motif packet: 720 training, 240 validation; checkout hashes verified; original Windows hashes matched after explicit CRLF reconstruction.")


In [ ]:
d4_paths = {
    "vague": "vague/validation_predictions.csv",
    "specified": "specified/validation_predictions_simple_signal_4mer_logreg.csv",
}
d4_actual_reports = {}
for d4_arm, d4_path in d4_paths.items():
    d4_predictions = pd.read_csv(StringIO(d4_archive["files"][d4_path]))
    if d4_arm == "vague":
        # Adapter only; raw archive bytes are unchanged.
        d4_predictions = d4_predictions.rename(columns={"probability_label_1": "probability"})
    d4_claimed = d4_parent["arms"][d4_arm]["metrics"]
    d4_report = audit_predictions(d4_historical, d4_predictions, claimed_metrics=d4_claimed)
    assert not d4_report["claim_disagreements"]
    assert d4_report["cross_split_group_count"] == 0
    assert d4_report["cross_split_duplicate_sequence_count"] == 0
    assert len(d4_predictions) == 240
    d4_actual_reports[d4_arm] = d4_report
    print(d4_arm, "· exact validation IDs; groups/duplicate sequences do not cross")
    print("ROC-AUC:", round(d4_report["metrics"]["roc_auc"], 7), "AP:", round(d4_report["metrics"]["average_precision"], 7))
print("Neither run reproduced the authored fixture's errors.")
print("Different features/search/objectives: this does not isolate prompt quality.")
print("Training correctness, biological scope and access history still require manual evidence.")


### Evidence → consequence → repair

Discuss one finding with your partner:

- **Evidence:** `group-a` crosses train/validation. **Consequence:** independent-group
  generalization is unsupported. **Repair:** for this separate exercise, obtain enough
  data, split by group, refit and evaluate on development data. Preserve the main-course manifest.
- **Evidence:** vocabulary fitted on all four rows. **Consequence:** validation inputs
  informed learned preprocessing. **Repair:** fit the pipeline on training only; this
  finding alone does not quantify its effect on the score.
- **Evidence:** probabilities are reversed by ID. **Consequence:** 100% positional
  accuracy attaches predictions to the wrong samples. **Repair:** exact one-to-one join;
  ID-aligned accuracy is 0%, without repairing the other defects.

**STOP:** no model fitting, test scoring or new agent service is needed. Complete the
[audit checklist](https://github.com/palexiou/ML_Course_2026/blob/main/instructor/audit-checklist.html)
with PASS / FAIL / UNVERIFIED, then defend one supported sentence. The Agentomics slot
uses the agreed audit-discussion fallback; no platform demo has been verified.
